# Testing Model Outputs
Alex has run some inferences we now need to test to see if they match what is already produced and available in the fafb dataset

In [2]:
from load_data.connect_clients import connect_cave_client, connect_flywire_client #Note move this to the main repo level to run, as a module
ENV_PATH = ".env"

# Connect to CAVE client
cave_client = connect_cave_client(ENV_PATH)

# Connect to FlyWire client
flywire_client = connect_flywire_client(ENV_PATH)

/opt/anaconda3/envs/fau_connectomics/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CAVE token already exists in your account. No need to add it again.
FLYWIRE token already exists in your account. No need to add it again.


## Load in JSON file

In [3]:
test_json = "./data/skeleton_test_t1_p1_fixed.json"
# Load in JSON file as a dataframe
import pandas as pd
import json

with open(test_json, 'r') as file:
    data = json.load(file)

df = pd.DataFrame(data)
display(df.head())

,_id,synapse_id,prediction
0,{'$oid': '6800c80f380cc9ed5a879215'},98373,"[2.1970388843328692e-05, 0.9999692440032959, 9..."
1,{'$oid': '6800c80f380cc9ed5a879216'},98415,"[2.5887402443913743e-05, 0.9999715089797974, 2..."
2,{'$oid': '6800c80f380cc9ed5a879217'},99167,"[5.943149062659359e-06, 0.9999641180038452, 1...."
3,{'$oid': '6800c80f380cc9ed5a879218'},99242,"[9.46293948800303e-06, 0.9999514818191528, 3.5..."
4,{'$oid': '6800c80f380cc9ed5a879219'},304977,"[0.028452927246689796, 0.8848700523376465, 0.0..."


## Load in json input file

# Check out Cave Client tables


In [4]:
cave_client.materialize.get_tables()

['hierarchical_neuron_annotations',
 'neuron_information_v2',
 'synapses_nt_v1',
 'nuclei_v1',
 'proofread_neurons',
 'fly_synapses_neuropil_v6']

In [5]:
# "x":438817,"y":164242,"z":217400

centre_point = [438817, 164242, 217400]
min_bounding_box = [438817 - 1000, 164242 - 1000, 217400 - 1000]
max_bounding_box = [438817 + 1000, 164242 + 1000, 217400 + 1000]
bounding_box = [min_bounding_box, max_bounding_box]

synapse_table = cave_client.info.get_datastack_info('synapses_nt_v1')
# df=cave_client.materialize.query_table(synapse_table,
#                                   filter_spatial_dict = {'post_pt_position': bounding_box})
# 
# 
# df=cave_client.materialize.synapse_query(
#                                   bounding_box=bounding_box)
# df
cave_client.materialize.synapse_query("synapses_nt_v1"
                                      )

HTTPError: 400 Client Error: BAD REQUEST for url: https://global.daf-apis.com/info/api/v2/datastack/full/synapses_nt_v1 content: b'{\n  "data": {\n    "auth_dataset": null,\n    "resource_namespace": "datastack",\n    "table_id": "synapses_nt_v1"\n  },\n  "error": "invalid_table_id",\n  "message": "Invalid table_id for service"\n}\n'

In [ ]:
cave_client.materialize.get_tables()

In [ ]:
cave_client.materialize.get_table_metadata(
    table_name='synapses_nt_v1',
)

# View a synapse

In [ ]:
from cloudvolume import CloudVolume
import numpy as np
import daisy
import napari

def validate_locs_nm(locs_nm, voxel_size, volume_shape_vox, crop_size_vox):
    """
    Validate and filter `locs_nm` to ensure they produce valid crops within the volume.

    Args:
        locs_nm (list of tuples): Locations in nanometers (Z, Y, X).
        voxel_size (tuple): Voxel size in nm (Z, Y, X).
        volume_shape_vox (tuple): Shape of the CloudVolume in voxels (X, Y, Z).
        crop_size_vox (tuple): Size of the crop in voxels (Z, Y, X).

    Returns:
        list: Validated and filtered `locs_nm` within volume bounds.
    """
    valid_locs = []

    # Convert volume shape to nanometers
    volume_extent_nm = tuple(vs * sz for vs, sz in zip(voxel_size[::-1], volume_shape_vox))

    crop_half_nm = tuple((cs * vs) // 2 for cs, vs in zip(crop_size_vox, voxel_size))

    for loc in locs_nm:
        z_nm, y_nm, x_nm = loc

        in_bounds = (
            crop_half_nm[0] <= z_nm < (volume_extent_nm[2] - crop_half_nm[0]) and
            crop_half_nm[1] <= y_nm < (volume_extent_nm[1] - crop_half_nm[1]) and
            crop_half_nm[2] <= x_nm < (volume_extent_nm[0] - crop_half_nm[2])
        )

        if in_bounds:
            valid_locs.append(loc)
        else:
            print(f"Skipping out-of-bounds loc (nm): {loc}")

    return valid_locs


def get_fafb_v14_voxels(
    locs_nm,
    voxel_size=(40, 4, 4),  # CloudVolume native voxel size
    size=(16, 160, 160),    # in voxel units
    precomputed_path="https://storage.googleapis.com/neuroglancer-fafb-data/fafb_v14/fafb_v14_orig/",
):
    vol = CloudVolume(precomputed_path, mip=0, cache=True, parallel=False)
    size = daisy.Coordinate(size)
    shape = vol.shape  # (X, Y, Z)

    # Convert locations in nanometers to voxel coordinates at native resolution
    locs_vox = [
        (
            int(z_nm / voxel_size[0]),
            int(y_nm / voxel_size[1]),
            int(x_nm / voxel_size[2])
        )
        for (z_nm, y_nm, x_nm) in locs_nm
    ]

    raw = []

    for loc in locs_vox:
        loc = daisy.Coordinate(loc)  # Z, Y, X
        start = loc - (size / 2)
        roi = daisy.Roi(start, size)

        z0, y0, x0 = map(int, roi.get_begin())
        sz, sy, sx = map(int, size)
        z1, y1, x1 = z0 + sz, y0 + sy, x0 + sx

        # Convert to CloudVolume order → X, Y, Z
        x0c, y0c, z0c = x0, y0, z0
        x1c, y1c, z1c = x1, y1, z1

        if not (
            0 <= x0c < shape[0] and x1c <= shape[0] and
            0 <= y0c < shape[1] and y1c <= shape[1] and
            0 <= z0c < shape[2] and z1c <= shape[2]
        ):
            print(f"Out-of-bounds: loc={loc}, range=({x0c}:{x1c}, {y0c}:{y1c}, {z0c}:{z1c})")
            chunk = np.zeros((sz, sy, sx), dtype=np.uint8)
        else:
            try:
                chunk = vol[x0c:x1c, y0c:y1c, z0c:z1c]
                chunk = np.asarray(chunk)

                if chunk.ndim == 4 and chunk.shape[-1] == 1:
                    chunk = np.squeeze(chunk, axis=-1)

                chunk = chunk.transpose(2, 1, 0)  # (Z, Y, X)
            except Exception as e:
                print(f"Error at {loc}: {e}")
                chunk = np.zeros((sz, sy, sx), dtype=np.uint8)

        raw.append(chunk.astype(np.float32))

    raw = np.stack(raw)
    raw_normalized = raw / 255.0 * 2.0 - 1.0
    return raw, raw_normalized


# Set voxel size and crop size
voxel_size = (40, 4, 4)
crop_size_vox = (16, 160, 160)

# Example locations (some may be invalid)
#z y x
locs_nm = [(214080, 162452, 437042), (214400, 163690, 436761), (212360, 160857, 438812), (213200, 160988, 437477), (206240, 142887, 451693), (206480, 142469, 452441), (203200, 127946, 417736), (219920, 148945, 437094), (215480, 161447, 438157), (216440, 165325, 435448), (219800, 145680, 438400), (190120, 135854, 449539), (190680, 135110, 451904), (178880, 138534, 461270), (219560, 145662, 439271), (208520, 157703, 444303), (207240, 141157, 451531), (215280, 165339, 435122), (220320, 148466, 440063), (206160, 141130, 438888), (199240, 127005, 418373), (187520, 130316, 456325), (206640, 141202, 450800), (208360, 143732, 437101), (205600, 142319, 437770), (187840, 133394, 450664), (187960, 134227, 449626), (181720, 136846, 465244), (199880, 127893, 419062), (199680, 129860, 419688), (220840, 148560, 439079), (221960, 147025, 439903), (219960, 146385, 437533), (195720, 129139, 423619), (202640, 128987, 417726), (181280, 137342, 462296), (199560, 127105, 416967), (209520, 159910, 441919), (209600, 158477, 441664), (208480, 157681, 442617), (188240, 160057, 343529), (164040, 161445, 341698), (184640, 159803, 343507), (175440, 162142, 349611)]

# Validate locations
# Get data
raw, raw_norm = get_fafb_v14_voxels(
    locs_nm=locs_nm,
    voxel_size=voxel_size,
    size=crop_size_vox
)
np.set_printoptions(threshold=np.inf)
print("Shape:", raw.shape)


print("Sample voxel [0,0,0,0]:", raw[0, 0, 0, 0])
print("Slice [0, Z mid]:")
print(raw[0, raw.shape[1] // 2])

## Trial 2: Using flywire

In [6]:
# Import skeleton ids
pth = "./data/test_data_from_alex/skeletons_fixed.json"
skeleton_df = pd.read_json(pth)
skeleton_df

,_id,skeleton_id,hemi_lineage_id,nt_known
0,{'$oid': '6800ba442b847cf645a963c1'},16,1,[acetylcholine]
1,{'$oid': '6800ba442b847cf645a963c2'},27,1,[acetylcholine]
2,{'$oid': '6800ba442b847cf645a963c3'},430,2,None
3,{'$oid': '6800ba442b847cf645a963c4'},734,3,[acetylcholine]
4,{'$oid': '6800ba442b847cf645a963c5'},949,1,[acetylcholine]
...,...,...,...,...
2799,{'$oid': '6800ba442b847cf645a96eb0'},11150632,90,[dopamine]
2800,{'$oid': '6800ba442b847cf645a96eb1'},11175371,90,[dopamine]
2801,{'$oid': '6800ba442b847cf645a96eb2'},11266651,90,[dopamine]
2802,{'$oid': '6800ba442b847cf645a96eb3'},11267636,90,[dopamine]


In [7]:
skeleton_ids = skeleton_df['skeleton_id'].tolist()
print("Skeleton IDs:", skeleton_ids)

Skeleton IDs: [16, 27, 430, 734, 949, 1165, 2076, 2115, 3133, 5714, 12578, 21999, 22132, 22277, 22422, 22594, 22744, 22906, 22976, 23005, 23134, 23432, 23512, 23569, 23597, 23829, 24251, 24622, 24726, 27048, 27246, 27295, 27611, 28876, 30434, 30571, 30791, 30891, 32214, 32399, 32793, 32801, 33903, 35246, 35447, 36108, 36390, 37212, 37235, 37250, 37935, 38885, 39139, 39254, 39668, 39682, 40306, 40637, 40749, 41308, 41578, 42421, 42927, 43539, 45242, 46493, 46800, 49026, 49865, 51080, 51886, 52106, 53631, 53671, 54072, 55085, 55125, 56424, 56623, 56983, 56995, 56999, 57003, 57007, 57011, 57015, 57019, 57023, 57035, 57039, 57047, 57051, 57059, 57063, 57067, 57071, 57076, 57080, 57089, 57094, 57098, 57102, 57106, 57114, 57122, 57126, 57130, 57134, 57138, 57142, 57146, 57154, 57158, 57166, 57171, 57175, 57179, 57192, 57196, 57200, 57204, 57208, 57212, 57216, 57220, 57224, 57232, 57236, 57241, 57246, 57254, 57258, 57266, 57270, 57274, 57278, 57307, 57311, 57319, 57323, 57333, 57337, 57341, 5

In [8]:
from fafbseg import flywire
import pymaid

tk = flywire.get_chunkedgraph_secret()
# Connect to the VFB's CATMAID
rm = pymaid.CatmaidInstance('https://fafb.catmaid.virtualflybrain.org/',
                            project_id=1, api_token=None)

root_ids = flywire.skid_to_id([16])
print("Root IDs:", root_ids)

INFO  : Global CATMAID instance set. Caching is ON. (pymaid)
                                                                       

Root IDs:   skeleton_id          flywire_id  confidence
0          16  720575940636873791        0.97


In [9]:
root = root_ids.iloc[0]['flywire_id']
print(root)

720575940636873791


In [10]:
# rt = str(root_id['flywire_id'].values)
rt = str(root)
pre_df = cave_client.materialize.query_table(
    table='synapses_nt_v1',
    filter_in_dict={'pre_pt_root_id': [rt]},
)

In [11]:
# Split 

## Predictions for single test neuron

### Get root predictions

In [12]:
root_id = "720575940607155890"
pre_df = cave_client.materialize.query_table(
    table='synapses_nt_v1',
    filter_in_dict={'pre_pt_root_id': [root_id]},
)
pre_df.columns


Index(['id', 'created', 'superceded_id', 'valid', 'connection_score',
       'cleft_score', 'gaba', 'ach', 'glut', 'oct', 'ser', 'da', 'valid_nt',
       'pre_pt_supervoxel_id', 'pre_pt_root_id', 'post_pt_supervoxel_id',
       'post_pt_root_id', 'pre_pt_position', 'post_pt_position'],
      dtype='object')

In [13]:
chosen_cols = ['id', 'pre_pt_supervoxel_id', 'pre_pt_root_id', 'pre_pt_position', 'gaba', 'ach', 'glut', 'oct', 'ser', 'da']
all_synapses = pre_df[chosen_cols].copy()
# Rename id to synapse_id
all_synapses.rename(columns={'id': 'synapse_fafb_id'}, inplace=True)

pre_df_filename = f"./data/test_data_from_alex/single_neuron/{root_id}_pre_synapses.parquet"

#Create a synapse_id as the index value, and move to the leftmost column
all_synapses['synapse_id'] = all_synapses.index

all_synapses.to_parquet(pre_df_filename, index=True)
all_synapses.head()

,synapse_fafb_id,pre_pt_supervoxel_id,pre_pt_root_id,pre_pt_position,gaba,ach,glut,oct,ser,da,synapse_id
0,508985,80293211209681776,720575940607155890,"[563932, 171384, 74560]",0.017340,0.064045,0.038678,0.000318,0.033393,0.846226,0
1,15429442,80293554739826794,720575940607155890,"[563400, 193200, 50560]",0.005293,0.310279,0.006239,0.003252,0.662614,0.012324,1
2,524492,80294173282369181,720575940607155890,"[564620, 230624, 74480]",0.049338,0.796867,0.026185,0.007607,0.007234,0.112770,2
3,1059342,80927904296794384,720575940607155890,"[600952, 254196, 74040]",0.028138,0.666006,0.005373,0.002674,0.007495,0.290314,3
4,27594708,80081830099025888,720575940607155890,"[551012, 157524, 67720]",0.204758,0.160491,0.156203,0.006174,0.074122,0.398252,4


In [15]:
all_synapses["pre_pt_position"].dtypes

dtype('O')

In [20]:
y_val = 227604
z_val = 83680
x_val = 672464
test = all_synapses.copy()
# Split pre_pt_position into a x, y, z columns
test[['pre_pt_x', 'pre_pt_y', 'pre_pt_z']] = test['pre_pt_position'].apply(
    lambda pos: pd.Series([pos[0], pos[1], pos[2]])
)
display(test.head())

# Find the row with the specified pre_pt_position
rows = test[(test['pre_pt_y'] == y_val) & (test['pre_pt_z'] == z_val)]
display(rows)

,synapse_fafb_id,pre_pt_supervoxel_id,pre_pt_root_id,pre_pt_position,gaba,ach,glut,oct,ser,da,synapse_id,pre_pt_x,pre_pt_y,pre_pt_z
0,508985,80293211209681776,720575940607155890,"[563932, 171384, 74560]",0.017340,0.064045,0.038678,0.000318,0.033393,0.846226,0,563932,171384,74560
1,15429442,80293554739826794,720575940607155890,"[563400, 193200, 50560]",0.005293,0.310279,0.006239,0.003252,0.662614,0.012324,1,563400,193200,50560
2,524492,80294173282369181,720575940607155890,"[564620, 230624, 74480]",0.049338,0.796867,0.026185,0.007607,0.007234,0.112770,2,564620,230624,74480
3,1059342,80927904296794384,720575940607155890,"[600952, 254196, 74040]",0.028138,0.666006,0.005373,0.002674,0.007495,0.290314,3,600952,254196,74040
4,27594708,80081830099025888,720575940607155890,"[551012, 157524, 67720]",0.204758,0.160491,0.156203,0.006174,0.074122,0.398252,4,551012,157524,67720


,synapse_fafb_id,pre_pt_supervoxel_id,pre_pt_root_id,pre_pt_position,gaba,ach,glut,oct,ser,da,synapse_id,pre_pt_x,pre_pt_y,pre_pt_z
3755,206261537,78042304815920779,720575940607155890,"[432536, 227604, 83680]",0.051315,0.503017,0.405292,0.008962,0.001679,0.029736,3755,432536,227604,83680
20377,206261542,78042304815920779,720575940607155890,"[432496, 227604, 83680]",0.031723,0.544194,0.357697,0.017949,0.002955,0.045481,20377,432496,227604,83680


In [ ]:
# Load in the predictions from the json
